# SpiderNet data loading and MI-dimension-selection tutorial

See [README.md](README.md) for the execution order, upstream inputs, commands and paper mapping.
Run the unified entry point to save executed copies and local outputs. Scientific variants and their parameters are retained below.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scanpy as sc

# If the notebook is run from the project root, this is usually enough.
sys.path.append(str(Path.cwd()))

from SpiderNet.dataloading_unified import (
    prepare_processed_bundle_unified,
    preview_lr_corr_distribution,
)
from SpiderNet.utils import (
    get_default_cellchat_db,
    get_default_scseqcomm_db,
)


## Step 1. Basic user inputs

Edit the variables in this section first.


In [ ]:
from workflow_paths import DATA_ROOT, PROCESSED_DATA_DIR, OUTPUT, input_path, output_path, ensure_output
ensure_output() ## Set this to your local data directory
OUTPUT_DIR = PROCESSED_DATA_DIR ## Set this to your desired processed data directory

SPECIES = "human" ## Set this to your species, used for ligand-receptor database loading. Options are "human" or "mouse".

SAMPLE_COL = "samples" ## Set this to the column in adata.obs that defines the sample labels (i.e. which cells belong to the same sample)
CELL_CLASS_COL = "cell.types" ## Set this to the column in adata.obs that defines the cell type annotation
PYG_EXTRA_OBS_FIELDS = {"patients": "patients"} ## Set this to a dict of {new_field_name: adata.obs column name} for any extra adata.obs fields you want copied into each PyG sample object. For example, {"patients": "patients"} means that adata.obs["patients"] will also be stored in each PyG sample object under data.patients.

N_HVG = 1000 ## Number of top highly variable genes used as SpiderNet model input for training
N_HVG_LR = 2000 ## Number of top highly variable genes used to define the candidate ligand-receptor search space; LR pairs are selected only if their genes are within these top genes
NUM_NEIGHBORS = 10 ## Number of spatial neighbors

# Set this only after running the preview section below
LR_CORR_THRESHOLD = None ## Set this to the desired ligand-receptor correlation threshold after inspecting the preview plot in Step 4. Common choices are around 0.5, but it depends on the dataset and the preview plot.

# Optional advanced override:
# Leave this as None for the standard SpiderNet input convention:
#   - raw counts stored in layers["counts"]
#   - spatial coordinates stored in obsm["spatial"]
# If your input AnnData only has a pre-normalized layer, set it here
# (for example, "normalizedexp"), and the notebook will skip normalize/log1p.
PRENORMALIZED_LAYER = None


## Step 2. Standard SpiderNet input

This notebook assumes the standard SpiderNet-ready AnnData format:

- raw counts are stored in `layers["counts"]`
- spatial coordinates are stored in `obsm["spatial"]`
- samples are defined by `obs[SAMPLE_COL]`

Because this convention is expected by the tutorial, these settings are kept out of the main
user-input block above.

If your data does **not** contain raw counts and only contains pre-normalized expression,
set `PRENORMALIZED_LAYER` in **Step 1**. The notebook will then read that layer directly and
skip `normalize_total` and `log1p`.


In [ ]:
CELLCHAT_DB = get_default_cellchat_db(species=SPECIES)
SCSEQCOMM_DB = get_default_scseqcomm_db(species=SPECIES)

if PRENORMALIZED_LAYER is None:
    DATA_REPRESENTATION_CONFIG = {
        "expression_source": {"kind": "layer", "name": "counts"},
        "normalize_strategy": "always",
        "log1p": True,
        "remove_zero_count_cells": False,
    }
else:
    DATA_REPRESENTATION_CONFIG = {
        "expression_source": {"kind": "layer", "name": PRENORMALIZED_LAYER},
        "normalize_strategy": "never",
        "log1p": False,
        "remove_zero_count_cells": False,
    }


## Step 3. Build the base config

This config contains the common loading settings.
`lr_corr_threshold` is intentionally left unset until after the preview step.


In [ ]:
base_config = {
    "data_path_main": DATA_ROOT,
    "output_dir": OUTPUT_DIR,
    "ligand_receptor_filedir_cellchatdb": CELLCHAT_DB,
    "ligand_receptor_filedir_scSeqComm": SCSEQCOMM_DB,

    "sample_col": SAMPLE_COL,
    "cell_class_col": CELL_CLASS_COL,
    "pyg_obs_fields": PYG_EXTRA_OBS_FIELDS,

    "n_hvg": N_HVG,
    "n_hvg_lr": N_HVG_LR,
    "num_neighbors": NUM_NEIGHBORS,

    # Keep unset until after the preview
    "lr_corr_threshold": LR_CORR_THRESHOLD,
}

base_config.update(DATA_REPRESENTATION_CONFIG)

print("CellChat DB:", CELLCHAT_DB)
print("scSeqComm DB:", SCSEQCOMM_DB)
print("Base output dir:", OUTPUT_DIR)

pd.Series({
    "data_path_main": str(base_config["data_path_main"]),
    "output_dir": str(base_config["output_dir"]),
    "sample_col": base_config["sample_col"],
    "cell_class_col": base_config["cell_class_col"],
    "n_hvg": base_config["n_hvg"],
    "n_hvg_lr": base_config["n_hvg_lr"],
    "num_neighbors": base_config["num_neighbors"],
    "expression_source": str(base_config["expression_source"]),
    "normalize_strategy": base_config["normalize_strategy"],
}).to_frame("value")


## Step 4. Preview the LR-correlation density

Run this cell **before** setting `LR_CORR_THRESHOLD`.

This preview only runs the minimal preprocessing needed to reach the LR-correlation stage.
It does **not** run the full `prepare_processed_bundle_unified(...)` pipeline, and it only shows
the density plot needed for choosing `lr_corr_threshold`.


In [ ]:
preview = preview_lr_corr_distribution(base_config, show_plot=True)

if preview.get("plot_path") is not None:
    print("Preview plot saved to:", preview["plot_path"])


## Step 5. Set `lr_corr_threshold` and run the full loader

After inspecting the density plot above, set `LR_CORR_THRESHOLD` to the desired value (commonly around 0.5, but it depends on the dataset and the preview plot), then run this cell to execute the full unified loading pipeline.


In [ ]:
LR_CORR_THRESHOLD = 0.5

In [ ]:
if LR_CORR_THRESHOLD is None:
    raise ValueError(
        "Please set LR_CORR_THRESHOLD in Step 1 after inspecting the preview plot, "
        "then rerun Step 3 and this cell."
    )

config = dict(base_config)
config["lr_corr_threshold"] = float(LR_CORR_THRESHOLD)

bundle = prepare_processed_bundle_unified(config)


## Step 6. Inspect the processed outputs

In [ ]:
print("Output dir:", bundle["output_dir"])
print("Number of batches:", len(bundle["batch_cell_unique"]))
print("Total cells:", bundle["adata"].n_obs)
print("All genes:", len(bundle["genenames"]))
print("Training genes:", len(bundle["genenames_train"]))
print("Training LR pairs:", len(bundle["LR_list"]))
print("Number of cell types:", bundle["adata"].obs[CELL_CLASS_COL].nunique())
print("Cell types:", bundle["adata"].obs[CELL_CLASS_COL].unique())

bundle["adata"].obs.head()


## Step 7. Save HGSOC-specific sample metadata

This is intentionally kept outside the unified loader because it is clearly
HGSOC-specific downstream metadata formatting.


In [ ]:
metadata_sample = pd.read_csv(input_path(DATA_ROOT / "HGSOC_metadata.csv"), index_col=0)

real_samples = np.sort(bundle["adata"].obs[SAMPLE_COL].astype(str).unique())

metadata_use = metadata_sample.loc[real_samples, :].copy()
metadata_use["stage_index"] = metadata_use["stage"].astype("category").cat.codes
metadata_use["treatment_index"] = metadata_use["treatment"].astype("category").cat.codes
metadata_use["sites_binary_index"] = metadata_use["sites_binary"].astype("category").cat.codes
metadata_use["outcome_use"] = metadata_use["outcome"].replace({"Dead of disease": "Dead (disease)"})
metadata_use["outcome_use_index"] = metadata_use["outcome_use"].astype("category").cat.codes
metadata_use["samples"] = metadata_use.index

metadata_use.to_csv(Path(bundle["output_dir"]) / "metadata_sample.csv", index=False)
metadata_use.head()


## Step 8. Run MI dimension selection on the processed outputs

This section reuses the processed files that were just written to `bundle["output_dir"]`.
It does **not** rerun data loading or preprocessing.

By default, the MI-dimension-selection thresholds are chosen automatically based on the
number of retained LR pairs. You can leave the optional overrides below as `None` unless
you want to tune the heuristic manually.


In [ ]:
from IPython.display import display
import pandas as pd

from SpiderNet.io import load_processed_data
from SpiderNet.MI_dimension_selection import run_mi_dimension_selection

# Optional advanced overrides for MI dimension selection.
# Leave these as None to use the default automatic heuristic.
MI_DIM_LR_SPEARCOR_THRESHOLD = None
MI_DIM_MIN_CLIQUE_SIZE = None
MI_DIM_JACCARD_THRESHOLD = None
MI_DIM_SHOW_HEATMAP = True

processed = load_processed_data(bundle["output_dir"])

print("Processed directory:", bundle["output_dir"])
print("Number of batches:", len(processed.spidernet_data))
print("Number of retained LR pairs:", len(processed.lr_list))
print("Number of training genes:", len(processed.genenames_train))

mi_dim_results = run_mi_dimension_selection(
    processed=processed,
    lr_list=processed.lr_list,
    output_dir=bundle["output_dir"],
    lr_spearcor_threshold=MI_DIM_LR_SPEARCOR_THRESHOLD,
    min_clique_size=MI_DIM_MIN_CLIQUE_SIZE,
    jaccard_thr=MI_DIM_JACCARD_THRESHOLD,
    show=MI_DIM_SHOW_HEATMAP,
)

mi_dim_summary_df = pd.DataFrame(
    [
        {
            "recommended_dim_envir": mi_dim_results["recommended_dim_envir"],
            "num_merged_subsets": mi_dim_results["num_merged_subsets"],
            "subset_sizes": mi_dim_results["subset_sizes"],
            "num_lr_pairs": mi_dim_results["num_lr_pairs"],
            "num_graph_edges": mi_dim_results["num_graph_edges"],
            "num_maximal_cliques_filtered": mi_dim_results["num_maximal_cliques_filtered"],
            "effective_min_clique_size": mi_dim_results["effective_min_clique_size"],
            "num_unassigned_lr_pairs": mi_dim_results["num_unassigned_lr_pairs"],
            "lr_spearcor_threshold": mi_dim_results["lr_spearcor_threshold"],
            "jaccard_thr": mi_dim_results["jaccard_thr"],
            "warning": mi_dim_results["warning"],
        }
    ]
)

display(mi_dim_summary_df)

print(f"Recommended dim_envir: {mi_dim_results['recommended_dim_envir']}")
print(f"Heatmap saved to: {mi_dim_results['heatmap_path']}")
print(f"Summary JSON saved to: {mi_dim_results['summary_path']}")
print(f"Subset summary CSV saved to: {mi_dim_results['subset_summary_path']}")
